[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SubSurfObs/observatory_notebooks/blob/main/02_phase_picking/index.ipynb)

# 02 — Automated Phase Picking with SeisBench

This notebook applies the [PhaseNet](https://doi.org/10.1093/gji/ggz233) deep-learning
picker (via [SeisBench](https://seisbench.readthedocs.io)) to multi-network waveform data,
then associates picks into events using [PyOcto](https://github.com/yetinam/pyocto).

**Prerequisites:** Run [notebook 01](../01_fdsn_access/index.ipynb) first,
or use the committed data files in `data/`.

**Worked example:** Mw ~3 earthquake near Moe, Victoria, 3 February 2026.


<table style="border:1px solid #BFC3D1;border-radius:6px;padding:0.5rem 1rem;margin-bottom:1rem;font-size:0.9em;background:#f9f9fb;border-collapse:separate;">
<tr><td style="padding:2px 12px 2px 0"><strong>Authors</strong></td><td>Dan Sandiford&nbsp;<a href="https://orcid.org/0000-0002-2207-6837"><img src="https://orcid.org/sites/default/files/images/orcid_16x16.png" alt="ORCID" style="vertical-align:middle"> 0000-0002-2207-6837</a></td></tr>
<tr><td style="padding:2px 12px 2px 0"><strong>Institution</strong></td><td>University of Melbourne — Subsurface Observatory</td></tr>
<tr><td style="padding:2px 12px 2px 0"><strong>Funding</strong></td><td>AuScope / National Collaborative Research Infrastructure Strategy (NCRIS)</td></tr>
<tr><td style="padding:2px 12px 2px 0"><strong>Data</strong></td><td>VW network &middot; DOI <a href="https://doi.org/10.7914/8csc-8z27">10.7914/8csc-8z27</a></td></tr>
<tr><td style="padding:2px 12px 2px 0"><strong>Notebook DOI</strong></td><td><em>in preparation (Zenodo)</em></td></tr>
<tr><td style="padding:2px 12px 2px 0"><strong>Licence</strong></td><td><a href="https://creativecommons.org/licenses/by/4.0/">CC BY 4.0</a></td></tr>
</table>

In [ ]:
# ── Colab detection + install ────────────────────────────────
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    import subprocess
    subprocess.run([sys.executable, '-m', 'pip', 'install',
                    'obspy', 'seisbench', 'pyocto', 'torch'], check=True)
    subprocess.run(['wget', '-q',
        'https://raw.githubusercontent.com/SubSurfObs/observatory_notebooks/main/utils.py'],
        check=True)
    # Also fetch the example data if not present
    from pathlib import Path
    Path('data').mkdir(exist_ok=True)
    for fname in ['ex01.mseed', 'ex01.xml']:
        if not Path(f'data/{fname}').exists():
            subprocess.run(['wget', '-q', '-P', 'data',
                'https://raw.githubusercontent.com/SubSurfObs/observatory_notebooks/'
                f'main/02_phase_picking/data/{fname}'], check=True)


In [ ]:
# ── Imports ──────────────────────────────────────────────────
from pathlib import Path
import sys

for p in [Path.cwd().parent, Path.cwd()]:
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

import numpy as np
import matplotlib.pyplot as plt
from obspy import read, read_inventory, UTCDateTime
import torch
import seisbench.models as sbm
import pyocto

from utils import (
    Provider, collect_stations_and_waveforms, convert_to_catalog,
    plot_station_picks_panel,
    FDSN_UOM, FDSN_AUSPASS,
)
print(f'SeisBench {sbm.__version__ if hasattr(sbm, "__version__") else "ok"}  |  '
      f'PyTorch {torch.__version__}')


## Load waveform data

Try loading from the committed `data/` files first; fall back to live FDSN
download if they are absent (e.g. first run in Colab without the wget above).


In [ ]:
data_dir   = Path('data')
mseed_path = data_dir / 'ex01.mseed'
xml_path   = data_dir / 'ex01.xml'

EVENT_LAT  = -38.18
EVENT_LON  = 146.35
EVENT_TIME = UTCDateTime('2026-02-03T08:12:00')
T_START    = EVENT_TIME - 30
T_END      = EVENT_TIME + 90

if mseed_path.exists() and xml_path.exists():
    print('Loading from disk...')
    st      = read(str(mseed_path))
    inv_all = read_inventory(str(xml_path))
else:
    print('Data not found locally — fetching from FDSN...')
    from obspy.core.inventory import Inventory
    st, selections = collect_stations_and_waveforms(
        providers=[Provider('UoM', FDSN_UOM), Provider('AusPass', FDSN_AUSPASS)],
        requested_networks=['VW', 'OZ'],
        latitude=EVENT_LAT, longitude=EVENT_LON, maxradius=3.0,
        starttime=T_START, endtime=T_END,
        channel='*H*,*N*', attach_response=True,
    )
    inv_all = Inventory()
    for sel in selections.values():
        inv_all += sel.inventory

print(f'Loaded: {len(st)} traces  |  {len(list(inv_all.get_contents()["stations"]))} stations')


## Pre-processing

Resample to 100 Hz, detrend, taper, and bandpass filter before picking.
PhaseNet was trained on 100 Hz data — resampling is required.


In [ ]:
# Work on a copy so the raw stream is preserved
st_proc = st.copy()
st_proc.resample(100.0)
st_proc.detrend('linear')
st_proc.taper(0.05)
st_proc.filter('bandpass', freqmin=2.0, freqmax=40.0, corners=4, zerophase=True)
st_proc.merge(fill_value='interpolate')
print(st_proc)


## Phase picking — PhaseNet

Load the pretrained PhaseNet model and run it across all stations.
Picks above the probability thresholds are retained.


In [ ]:
# Tunable parameters
PHASENET_MODEL = 'original'
P_THRESHOLD    = 0.25
S_THRESHOLD    = 0.25
N_PICKS        = 6      # picks kept per phase per station
N_PS_PICKS     = 3      # must have both P+S


In [ ]:
picker = sbm.PhaseNet.from_pretrained(PHASENET_MODEL)
picks  = picker.classify(
    st_proc,
    P_threshold=P_THRESHOLD,
    S_threshold=S_THRESHOLD,
).picks
print(f'Total picks: {len(picks)}')


In [ ]:
# Summarise picks per station
import pandas as pd
from collections import Counter

phase_counts = Counter((p.trace_id.split('.')[1], p.phase) for p in picks)
summary = (pd.DataFrame([
    {'station': sta, 'phase': ph, 'count': n}
    for (sta, ph), n in sorted(phase_counts.items())
])
    .pivot_table(index='station', columns='phase', values='count', fill_value=0)
    .reset_index()
)
display(summary)


## Event association — PyOcto

Use PyOcto's grid-based associator to group picks into candidate events.
We use a simple constant-velocity model appropriate for the upper crust
of southeastern Australia.


In [ ]:
# Simple two-layer Vp model (uppermost crust, SE Australia)
Vp = 6.0   # km/s
Vp_Vs = 1.73

# Bounding box: centre on the station network centroid
lats_sta, lons_sta = [], []
for net in inv_all:
    for sta in net:
        if sta.latitude and sta.longitude:
            lats_sta.append(sta.latitude)
            lons_sta.append(sta.longitude)

lat_c = sum(lats_sta) / len(lats_sta)
lon_c = sum(lons_sta) / len(lons_sta)
margin = 2.0   # degrees
lat_box = (lat_c - margin, lat_c + margin)
lon_box = (lon_c - margin, lon_c + margin)
print(f'Search box: lat {lat_box}, lon {lon_box}')


In [ ]:
associator = pyocto.OctoAssociator.from_area(
    lat=lat_box,
    lon=lon_box,
    zlim=(0, 50),
    time_before=20,
    velocity_model=pyocto.VelocityModel0D(Vp, Vp / Vp_Vs),
    n_picks=N_PICKS,
    n_p_and_s_picks=N_PS_PICKS,
)


In [ ]:
stations_df = associator.inventory_to_df(inv_all)
# PyOcto may return duplicate station entries; keep unique by id
stations_df = stations_df.drop_duplicates(subset='id').reset_index(drop=True)

events, assignments = associator.associate_seisbench(picks, stations_df)
associator.transform_events(events)

print(f'Events found: {len(events)}')
if len(events):
    print(events[['latitude', 'longitude', 'depth', 'time']].to_string())


## Visualise picks

Plot waveforms for the stations with the clearest P picks, aligned on the
earliest P arrival.


In [ ]:
stations_to_plot = (
    assignments
    .query("phase == 'P'")
    .sort_values('time')
    .station
    .str.split('.').str[1]   # extract station code from NET.STA.LOC
    .drop_duplicates()
    .tolist()
)

plot_station_picks_panel(
    st_proc,
    stations_to_plot[:6],
    assignments=assignments,
    channel='*Z',
)


## Save outputs

Export the pick assignments as CSV and the located catalog as QuakeML
for use in subsequent notebooks (instrument correction, magnitude estimation).


In [ ]:
out_dir = Path('data')
out_dir.mkdir(exist_ok=True)

# Add human-readable UTC time column
assignments['time_utc'] = assignments['time'].apply(UTCDateTime)
assignments.to_csv(out_dir / 'ex01_assignments.csv', index=False)
print(f'Saved {out_dir / "ex01_assignments.csv"}')

cat = convert_to_catalog(events, assignments)
cat.write(str(out_dir / 'ex01_cat.xml'), format='QUAKEML')
print(f'Saved {out_dir / "ex01_cat.xml"}')


---

**Previous:** [01 — FDSN Data Access](../01_fdsn_access/index.ipynb)  
**Next:** 03 — Earthquake Location *(in preparation)*

**Data source:** University of Melbourne Seismic Network (VW),
DOI [10.7914/8csc-8z27](https://doi.org/10.7914/8csc-8z27).  
**Licence:** [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/).
